In [2]:
# Kiểm tra nhanh các thư viện cơ bản (không bắt buộc tất cả phải có)
import sys, json, math, random, os, time
import numpy as np

try:
    import sklearn
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_OK = True
except Exception as e:
    SKLEARN_OK = False
    print("Thiếu scikit-learn. Các phần dùng sklearn sẽ không chạy:", e)

try:
    import networkx as nx
    NETWORKX_OK = True
except Exception as e:
    NETWORKX_OK = False
    print("Thiếu networkx. Các phần đồ thị sẽ không chạy:", e)

# Các phần tùy chọn (có thể không có internet nên sẽ fail ở đây, cứ để False nếu không)
try:
    import torch
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False

try:
    import transformers
    TRANSFORMERS_OK = True
except Exception as e:
    TRANSFORMERS_OK = False

print("SKLEARN_OK =", SKLEARN_OK, "| NETWORKX_OK =", NETWORKX_OK, "| TORCH_OK =", TORCH_OK, "| TRANSFORMERS_OK =", TRANSFORMERS_OK)

SKLEARN_OK = True | NETWORKX_OK = True | TORCH_OK = True | TRANSFORMERS_OK = True


## 12) GNN (Graph Neural Networks) – Ứng dụng: **Phát hiện cộng đồng** (Karate Club)

**Bối cảnh:** Mạng xã hội (graph) – ta muốn dự đoán **cộng đồng** của mỗi người dùng.
Ở đây ta dùng đồ thị **Zachary Karate Club** kinh điển.

Ta xây một **GNN tổng quát kiểu GraphSAGE (mean)**:
- Mỗi lớp: `h_v^{(k+1)} = MLP( concat( h_v^{(k)}, mean_{u∈N(v)} h_u^{(k)} ) )`
- Train bằng cross-entropy từ vài **nhãn mồi** (semi-supervised).

Mục tiêu: thấy quy trình GNN cơ bản mà **không phụ thuộc framework DL**.

In [3]:
import numpy as np

if not 'NETWORKX_OK' in globals() or not NETWORKX_OK:
    print("Cần networkx cho ví dụ này.")
else:
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    A = nx.to_numpy_array(G)  # adjacency (0/1)

    # Nhãn ground truth theo club ('Mr. Hi' vs 'Officer')
    clubs = np.array([0 if G.nodes[i]['club'] == 'Mr. Hi' else 1 for i in range(n)])

    rng = np.random.default_rng(0)
    # Đặc trưng khởi tạo (d_x = 16)
    X = rng.normal(0,1,size=(n,16)).astype(np.float32)

    # Chia tập train (10 node gán nhãn), val/test còn lại
    idx_all = np.arange(n)
    rng.shuffle(idx_all)
    idx_train = idx_all[:10]
    idx_test  = idx_all[10:]

    # GraphSAGE (mean) 2 lớp, numpy thuần
    def mean_agg(H, A):
        deg = A.sum(axis=1, keepdims=True) + 1e-8
        return (A @ H) / deg

    def relu(z): return np.maximum(z,0)
    def softmax(z):
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    # Tham số
    d_in, d_h, d_out = X.shape[1], 32, 2
    W1 = rng.normal(0, 0.1, size=(d_in*2, d_h))
    b1 = np.zeros((d_h,))
    W2 = rng.normal(0, 0.1, size=(d_h*2, d_out))
    b2 = np.zeros((d_out,))

    lr = 0.05
    epochs = 400

    def forward(X):
        N = X.shape[0]
        m1 = mean_agg(X, A)                      # [N, d_in]
        H1 = relu( X @ W1[:d_in] + m1 @ W1[d_in:] + b1 )  # concat bằng phép cộng tuyến tính tương đương
        m2 = mean_agg(H1, A)
        Z  = ( H1 @ W2[:d_h] + m2 @ W2[d_h:] + b2 )
        P  = softmax(Z)
        return H1, P

    def loss_and_grads(X, labels, idx):
        # cross-entropy trên idx
        N = X.shape[0]
        H1, P = forward(X)

        Y = np.zeros((N, d_out))
        Y[np.arange(N), labels] = 1.0

        mask = np.zeros((N,1))
        mask[idx] = 1.0
        L = - (mask * (Y * np.log(P+1e-9)).sum(axis=1, keepdims=True)).sum() / mask.sum()

        # Backprop (tối giản, không quá tối ưu; minh họa thôi)
        # dZ
        dZ = (P - Y) * mask  # [N,d_out]
        m2 = mean_agg(H1, A)

        # grads W2,b2
        dW2a = H1.T @ dZ      # phần với W2[:d_h]
        dW2b = m2.T @ dZ      # phần với W2[d_h:]
        db2  = dZ.sum(axis=0)

        # dH1 từ nhánh Z
        dH1 = dZ @ W2[:d_h].T
        dm2 = dZ @ W2[d_h:].T
        # dm2 = d(mean_agg(H1)) -> phân bố lại về H1
        # mean_agg: m2_i = sum_j A_ij H1_j / deg_i
        # gradient ngược: dH1_j += sum_i (A_ij/deg_i) * dm2_i
        deg = A.sum(axis=1, keepdims=True) + 1e-8
        back = (dm2 / deg)  # [N,d_h]
        dH1 += A.T @ back

        # ReLU
        dH1[H1<=0] = 0

        # Lớp 1
        m1 = mean_agg(X, A)
        dW1a = X.T @ dH1     # cho W1[:d_in]
        dW1b = m1.T @ dH1    # cho W1[d_in:]
        db1  = dH1.sum(axis=0)

        # Không cập nhật X (đặc trưng ngõ vào cố định)
        return float(L), dW2a, dW2b, db2, dW1a, dW1b, db1

    for ep in range(1, epochs+1):
        L, dW2a, dW2b, db2, dW1a, dW1b, db1 = loss_and_grads(X, clubs, idx_train)
        W2[:d_h] -= lr * dW2a; W2[d_h:] -= lr * dW2b; b2 -= lr * db2
        W1[:d_in] -= lr * dW1a; W1[d_in:] -= lr * dW1b; b1 -= lr * db1
        if ep % 50 == 0:
            _, P = forward(X)
            pred = P.argmax(axis=1)
            acc_train = (pred[idx_train] == clubs[idx_train]).mean()
            acc_test  = (pred[idx_test]  == clubs[idx_test]).mean()
            print(f"Epoch {ep:3d} | loss={L:.3f} | acc_train={acc_train:.2f} | acc_test={acc_test:.2f}")


Epoch  50 | loss=0.004 | acc_train=1.00 | acc_test=0.96
Epoch 100 | loss=0.001 | acc_train=1.00 | acc_test=0.92
Epoch 150 | loss=0.001 | acc_train=1.00 | acc_test=0.92
Epoch 200 | loss=0.001 | acc_train=1.00 | acc_test=0.92
Epoch 250 | loss=0.000 | acc_train=1.00 | acc_test=0.92
Epoch 300 | loss=0.000 | acc_train=1.00 | acc_test=0.92
Epoch 350 | loss=0.000 | acc_train=1.00 | acc_test=0.92
Epoch 400 | loss=0.000 | acc_train=1.00 | acc_test=0.92
